# Getting started: failure propagation and robustness

This notebook introduces the **stationary** Probabilistic Building Model with
an eight-compartment lattice. It runs 20,000 Monte Carlo realizations for an
undamaged baseline and 20,000 for the forced loss of one corner compartment.
These small examples teach the workflow; they do not reproduce manuscript figures.

Install the package and the Python 3.12 paper requirements using the
[README](../README.md#getting-started). Select that environment as your kernel.
Run this notebook from the repository root or the `notebooks/` directory.
Standard plots require no external LaTeX. Each execution writes a fresh local
directory under `output/getting-started/`; existing runs are preserved.

## 1. Set up the example

Nodes represent redundant compartments and edges possible propagation routes.
We choose $p_f=0.02$ per node and $p_e=0.35$ per transmission attempt so that
the contrast between the two scenarios is visible with a small sample count.
These values differ from the manuscript's inputs.

The command-line simulator is invoked through the current kernel's Python,
so the same interpreter is used throughout. A single worker keeps this example
simple; the package supports larger parallel campaigns separately.

In [ ]:
%matplotlib inline
from contextlib import closing
from pathlib import Path
import json
import sqlite3
import subprocess
import sys
import tempfile

import matplotlib.pyplot as plt
import numpy as np
from pbm.network.builder import build_and_insert
from pbm.post.robustness import bernoulli_kl_robustness

current = Path.cwd().resolve()
REPO_ROOT = current if (current / 'pbm').is_dir() else current.parent
if not (REPO_ROOT / 'pbm').is_dir():
    raise RuntimeError('Run from the repository root or its notebooks directory.')

NX, NY, NZ = 2, 2, 2
N = NX * NY * NZ
PF_NODE, PE = 0.02, 0.35
SAMPLES, SEED, S_LIM = 20_000, 123, 2
output = REPO_ROOT / 'output' / 'getting-started'
output.mkdir(parents=True, exist_ok=True)
WORK = Path(tempfile.mkdtemp(prefix='example-', dir=output))
INPUT_DB, RESULTS_DB = WORK / 'input.db', WORK / 'results.db'
print('Results directory:', WORK.relative_to(REPO_ROOT).as_posix())

## 2. Build and inspect the network

A $2\times2\times2$ lattice has $N=8$ nodes and twelve physical adjacencies.
The builder uses undirected edges by default, with the same probability in
both directions. Set the example's probabilities in the input database, then
inspect the stored network rather than inferring its size from a plot.

In [ ]:
build_and_insert(NX, NY, NZ, INPUT_DB, truncate=False)
with closing(sqlite3.connect(INPUT_DB)) as connection:
    connection.execute('UPDATE nodes SET pf_node=?', (PF_NODE,))
    connection.execute('UPDATE edges SET pe=?', (PE,))
    connection.commit()
    nodes = connection.execute('SELECT id,x,y,z,pf_node FROM nodes ORDER BY id').fetchall()
    edge_count = connection.execute('SELECT COUNT(*) FROM edges').fetchone()[0]
assert len(nodes) == N and edge_count == 12
print(f'{len(nodes)} nodes, {edge_count} physical adjacencies')
print('Nodes: (id, x, y, z, spontaneous failure probability)')
for row in nodes:
    print(row)

## 3. Simulate a baseline and a damage scenario

In the **baseline**, every node has an independent spontaneous initial failure
indicator $F_i(0)\sim\mathrm{Bernoulli}(p_f)$.
In the **damage scenario**, the node at $(0,0,0)$ is forced to fail at the start;
all other nodes retain their spontaneous failure probabilities.

A newly failed node attempts each outgoing transmission once. Failures are
irreversible. The final count $N_f$ measures the extent of the resulting cascade.

Both scenarios use the same seed and realization-index streams, providing
common random numbers. Comparisons should use identical networks, probabilities,
sample counts and seeds, changing only the prescribed damage set.

In [ ]:
scenarios = [(0, 'Undamaged baseline', []),
             (1, 'Corner damage', ['--damage-coords', '0,0,0'])]
for scenario_id, label, damage_args in scenarios:
    command = [sys.executable, '-m', 'cli.cli_simulation',
               '--db', str(INPUT_DB), '--outdb', str(RESULTS_DB),
               '--samples', str(SAMPLES), '--threshold', str(S_LIM),
               '--seed', str(SEED), '--scenario-id', str(scenario_id),
               '--mc-workers', '1', *damage_args]
    result = subprocess.run(command, cwd=REPO_ROOT, capture_output=True,
                            text=True, encoding='utf8')
    (WORK / f'scenario-{scenario_id}.log').write_text(
        result.stdout + result.stderr, encoding='utf8')
    if result.returncode:
        raise RuntimeError(f'{label} failed: {result.stderr or result.stdout}')
    print(f'{label}: {SAMPLES:,} realizations completed')

## 4. Read and plot the collapse-size distributions

The probability mass function (PMF) is estimated from integer counts:

$$
\widehat P(N_f=n)=\frac{\text{count of realizations with }N_f=n}{M},
\qquad n=0,\ldots,N.
$$

Here $M$ is the number of realizations per scenario. We select runs by their
stored scenario identifiers and check that all PMF bins are present and sum to
$M$. The baseline can have $N_f=0$; forced corner damage makes that impossible
in the second scenario. Complete collapse is the last bin, $N_f=N$.

In [ ]:
counts, run_ids = {}, {}
with closing(sqlite3.connect(RESULTS_DB)) as connection:
    for scenario_id, label, _ in scenarios:
        run = connection.execute(
            'SELECT id,samples,seed,threshold,finished_at FROM runs '
            'WHERE scenario_id=? ORDER BY id DESC LIMIT 1', (scenario_id,)).fetchone()
        assert run and run[1:4] == (SAMPLES, SEED, S_LIM) and run[4]
        rows = connection.execute(
            'SELECT n_failed,count FROM pmf WHERE run_id=? ORDER BY n_failed',
            (run[0],)).fetchall()
        assert [row[0] for row in rows] == list(range(N + 1))
        histogram = np.array([row[1] for row in rows], dtype=np.int64)
        assert histogram.sum() == SAMPLES
        counts[scenario_id], run_ids[scenario_id] = histogram, run[0]
assert counts[1][0] == 0

fig, ax = plt.subplots(figsize=(8, 4))
for scenario_id, label, _ in scenarios:
    ax.plot(np.arange(N + 1), counts[scenario_id] / SAMPLES,
            marker='o', label=label)
ax.set(xlabel=r'Final failed compartments $N_f$', ylabel='Estimated probability',
       title=f'Illustrative eight-compartment model ({SAMPLES:,} samples per scenario)')
ax.set_xticks(np.arange(N + 1))
ax.grid(alpha=0.25)
ax.legend()
fig.tight_layout()
fig.savefig(WORK / 'collapse-size-distributions.png', dpi=160)
plt.show()

## 5. Define an adverse event and measure robustness

We define the adverse event as **at least two failed compartments**,
$N_f\ge S_{\mathrm{lim}}$ with $S_{\mathrm{lim}}=2$.
Let $p_0$ be its baseline probability and $p_j$ its probability after corner
damage. Calculate both from the same PMF bins; do not compare different adverse
events or use the forced initial failure itself as the damaged probability.

For the paper's scale $c=1$, the entropy-based robustness index is

$$
I_{\mathrm{rob}}=\exp[-D_{\mathrm{KL}}],\qquad
D_{\mathrm{KL}}=p_0\ln\frac{p_0}{p_j}
+(1-p_0)\ln\frac{1-p_0}{1-p_j}.
$$

The divergence is baseline to damaged, and the logarithms are natural.
Equal probabilities give an index of one; a larger change towards the adverse
event lowers it. The comparison assumes $p_j\ge p_0$; the package checks this
coherence condition. It does not silently replace an incoherent estimate.

In [ ]:
p0 = float(counts[0][S_LIM:].sum() / SAMPLES)
pj = float(counts[1][S_LIM:].sum() / SAMPLES)
robustness = bernoulli_kl_robustness(p0, pj)
assert robustness.accepted and robustness.value is not None
assert 0 <= p0 <= pj <= 1 and 0 <= robustness.value <= 1
print(f'Adverse event: N_f >= {S_LIM}')
print(f'Undamaged probability p0 = {p0:.5f}')
print(f'Corner-damage probability pj = {pj:.5f}')
print(f'Entropy-based robustness = {robustness.value:.5f}')

summary = {'shape': [NX, NY, NZ], 'pf_node': PF_NODE, 'pe': PE,
           'samples_per_scenario': SAMPLES, 'seed': SEED, 's_lim': S_LIM,
           'run_ids': run_ids, 'counts': {str(k): v.tolist() for k, v in counts.items()},
           'p0': p0, 'pj': pj, 'robustness': robustness.to_dict()}
(WORK / 'summary.json').write_text(json.dumps(summary, indent=2), encoding='utf8')

## Interpretation and next steps

The two probabilities and the index describe **this network, this damage set,
and this adverse event**. They are Monte Carlo estimates from a small example,
not exact probabilities or a structural safety certification. Small or zero
counts need more simulation to resolve rare events. We do not assign the
manuscript's low/medium/high categories to this example.

Try changing `S_LIM` to four and rerunning all cells to see why the adverse-event
definition matters. Increase `SAMPLES` to explore sampling stability. Every rerun
creates a new results directory; the saved logs, PMFs and summary remain available.

For the scientific formulation, see Cao and Beck (2026),
[*On the quantification of robustness and its thresholds*](https://doi.org/10.1016/j.strusafe.2026.102688),
and the repository's [analysis methods](../docs/analysis-methods.md).
Next, inspect the [curated paper results](../reproduction/README.md) or follow the
[Figures 3–5 guide](../docs/figures-3-5-reproduction.md). The paper notebooks use
much larger campaigns; inspect their settings before running them.

When reusing the software, cite the associated paper and software as stated
in the [README](../README.md#citation). Current licensing and proposed terms are
also described there.